USING LANGCHAIN

### 📘 **Introduction**

This notebook demonstrates how to build a **Retrieval-Augmented Generation (RAG)** system by combining semantic search with a large language model (LLM). The goal is to accurately answer user questions by retrieving relevant information from a dataset before passing it to the LLM.

The pipeline integrates three core components:

---

#### Semantic Retrieval with Sentence Transformers  
A **bi-encoder** is used to generate embeddings for a large set of questions.  
When a user submits a query, the system compares it to the embedded dataset and retrieves the most **semantically similar questions**.

---

#### Re-ranking with a Cross-Encoder  
The initial **top-k** results are refined using a **cross-encoder**, which performs pairwise scoring between the user's query and each candidate.  
This improves accuracy by ranking based on **contextual relevance**.

---

#### Answer Generation with LangChain + LLM  
After identifying the most relevant questions and associated documents, a **prompt is dynamically constructed**.  
This prompt includes:
- the original user query  
- the retrieved documents  

It is passed to a powerful **LLM (e.g., Google Gemini)** via **LangChain**, which generates a final, **context-aware answer**.


In [1]:
# %%
!pip install -q sentence-transformers datasets==2.16.0

from sentence_transformers import SentenceTransformer, CrossEncoder, util
from datasets import load_dataset
import torch

EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
CROSSENCODER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
TOP_K = 10        # initial retrieval
FINAL_TOP_K = 3   # final top-k after re-ranking

print("Loading embedding model...")
bi_encoder = SentenceTransformer(EMBEDDING_MODEL)

print("Loading cross-encoder for re-ranking...")
cross_encoder = CrossEncoder(CROSSENCODER_MODEL)

print("Downloading dataset from Hugging Face Hub...")
dataset = load_dataset("FreedomIntelligence/RAG-Instruct", split="train", trust_remote_code=True)

questions_dataset = [item["question"] for item in dataset]
print(f"✅ {len(questions_dataset)} questions loaded.")

# Generate embeddings once
print("Generating embeddings...")
embeddings_dataset = bi_encoder.encode(questions_dataset, convert_to_tensor=True)

def retrieve_and_rerank(user_question):
    # Embed the user's question
    user_embedding = bi_encoder.encode(user_question, convert_to_tensor=True)

    # Get initial top-k candidates using cosine similarity
    similarities = util.pytorch_cos_sim(user_embedding, embeddings_dataset)[0]
    top_k = torch.topk(similarities, k=TOP_K)
    top_k_indices = top_k.indices.tolist()

    # Prepare pairs for cross-encoder re-ranking
    candidate_pairs = [(user_question, questions_dataset[i]) for i in top_k_indices]

    print("Re-ranking with CrossEncoder...")
    rerank_scores = cross_encoder.predict(candidate_pairs)

    # Sort by cross-encoder score
    reranked = list(zip(top_k_indices, rerank_scores))
    reranked.sort(key=lambda x: x[1], reverse=True)

    # Prepare final outputs
    final_questions = []
    final_documents = []

    for idx, score in reranked[:FINAL_TOP_K]:
        question = dataset[idx]["question"]
        documents = dataset[idx]["documents"]
        if isinstance(documents, str):
            documents = [documents]

        final_questions.append(question)
        final_documents.extend(documents)

    return final_questions, final_documents

user_question = input("Enter your question: ")
#user_question = transcribed_text = transcribe_audio(audio_file)

final_questions, final_documents = retrieve_and_rerank(user_question)

print("\n✅ Final Top-K Similar Questions:")
for i, q in enumerate(final_questions, 1):
    print(f"{i}. {q}")

print("\n📄 Aggregated Related Documents:")
for i, doc in enumerate(final_documents, 1):
    print(f"Document {i}: {doc.strip()[:300]}...")  # Preview first 300 characters




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 90.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not 

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

✅ 40541 questions loaded.
Generating embeddings...
Enter your question: jungle book
Re-ranking with CrossEncoder...

✅ Final Top-K Similar Questions:
1. Describe the cultural impact and legacy of the 1967 Disney film 'The Jungle Book' on animation and popular culture.
2. Describe Hathi's role and personality in Kipling's 'The Jungle Book' stories.
3. Identify two animals who play crucial roles in Mowgli's life in the jungle.

📄 Aggregated Related Documents:
Document 1: decided to make the story more straightforward, as the novel is very episodic, with Mowgli going back and forth from the jungle to the Man-Village, and Peet felt that Mowgli returning to the Man-Village should be the ending for the film. Following suggestions, Peet also created two original characte...
Document 2: and settings. In 2016, a Baloo figure was released for the console and later on mobile versions of "Disney Infinity 3.0" (2015), which required a downloadable content update to use. Although the figure was rele

In [3]:
!pip install langchain

In [4]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.1 MB/s eta 0:00:00
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [5]:
API_KEY=""

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=API_KEY)

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


# Prompt
template = """
You are a helpful and knowledgeable assistant. Your task is to answer the user's question using the provided context.

If the context is empty or does not contain sufficient information to answer the question reliably, respond by saying:
"I'm sorry, but I don't have enough information to accurately answer your question."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

# Función para preparar el input al modelo
def build_inputs(user_question):
    _, documents = retrieve_and_rerank(user_question)
    context = "\n\n".join(documents) if documents else ""
    return {
        "context": context,
        "question": user_question
    }


chain = RunnableLambda(build_inputs) | prompt | llm

query = "Did Captain McGonagle relinquish control of the Liberty before or after commanding it for 17 hours following the attack?"
query = "soy gay"
result = chain.invoke(query)

print("\n🧠 Response:\n")
print(result.content)



Re-ranking with CrossEncoder...

🧠 Response:

I'm sorry, but I don't have enough information to accurately answer your question.
